# NB-3: MuSiQue — Multi-Hop + L3 Isolation + 8B Variance Seeds

Runs CSAM on MuSiQue multi-hop QA at 100 questions. Also runs without L3 to isolate
the knowledge-graph hop contribution (PB-13).

**Runs:**
- Seed 42 — all 4 models WITH L3 (full CSAM)
- Seed 123 — 8B WITH L3
- Seed 456 — 8B WITH L3
- Seed 42 — 8B WITHOUT L3 (L3 ablation; quantifies KG contribution)

**Output directory:** `results/nb3_musique/`  
**Time estimate:** ~15 min per model run at 100Q

**Just run all cells top-to-bottom. No interaction needed after cell 3.**

## Step 1 — Install dependencies & clone repo

In [ ]:
import os, subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'sentence-transformers', 'hnswlib', 'python-dotenv', 'groq', 'requests'], check=True)
print('Deps installed')

REPO = 'https://github.com/Lamaq-Mujpurwala/CSAM-IPD-HALH.git'
REPO_DIR = '/kaggle/working/CSAM-IPD-HALH' if os.path.exists('/kaggle') else '/content/CSAM-IPD-HALH'

if os.path.exists(REPO_DIR):
    r = subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', 'main'],
                       capture_output=True, text=True)
    print(r.stdout.strip() or 'Already up to date')
    if r.returncode != 0:
        print('[WARN] pull failed:', r.stderr[:200])
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO, REPO_DIR], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

commit = subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True, cwd=REPO_DIR)
print(f'Commit: {commit.stdout.strip()}')
print(f'Working dir: {os.getcwd()}')

## Step 2 — Set API keys

**Kaggle:** Secrets → `GROQ_API_KEY` (+ optionally `GROQ_API_KEY_2` … `GROQ_API_KEY_5`)  
**Colab:** Left sidebar key icon → same secrets

In [ ]:
import os

def _load_secret(name: str) -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v: return v
    except Exception: pass
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception: pass
    return os.environ.get(name, '')

env_lines = []
key = _load_secret('GROQ_API_KEY')
if not key:
    raise RuntimeError('GROQ_API_KEY not found — add it to Kaggle/Colab Secrets')
os.environ['GROQ_API_KEY'] = key
env_lines.append(f'GROQ_API_KEY={key}')
print('GROQ_API_KEY loaded')

for i in range(2, 10):
    k = _load_secret(f'GROQ_API_KEY_{i}')
    if k:
        os.environ[f'GROQ_API_KEY_{i}'] = k
        env_lines.append(f'GROQ_API_KEY_{i}={k}')
        print(f'GROQ_API_KEY_{i} loaded')

with open('.env', 'w') as f:
    f.write('\n'.join(env_lines) + '\n')
print(f'\n.env written with {len(env_lines)} key(s)')

## Step 3 — Configure

In [ ]:
import os

# ── EDIT THESE IF NEEDED ────────────────────────────────────────────────────
N_QUESTIONS    = 100  # questions per run (50 for quick test, 100 for publication)
CHECKPOINT_DIR = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
# ─────────────────────────────────────────────────────────────────────────────

DATASET = os.path.join(REPO_DIR, 'csam_project', 'benchmarks', 'data', 'musique_dev.jsonl')
OUT_DIR = os.path.join(REPO_DIR, 'results', 'nb3_musique')
os.makedirs(OUT_DIR, exist_ok=True)

print(f'Dataset:    {DATASET}')
print(f'Dataset OK: {os.path.exists(DATASET)}')
print(f'Output dir: {OUT_DIR}')
print(f'Questions:  {N_QUESTIONS} per run')

if not os.path.exists(DATASET):
    data_dir = os.path.dirname(DATASET)
    print(f'[WARN] Dataset missing. Files in {data_dir}:')
    try: print(os.listdir(data_dir))
    except Exception as e: print(f'  {e}')

## Step 4 — Run 1: All 4 models WITH L3, seed 42, 100Q
Primary multi-model comparison with full CSAM (3-tier including knowledge graph).

In [ ]:
import subprocess, sys, os
os.chdir(REPO_DIR)

cmd = [
    sys.executable, '-m', 'csam_project.benchmarks.benchmark_musique',
    '--all',
    '--questions', str(N_QUESTIONS),
    '--dataset', DATASET,
    '--seed', '42',
    '--checkpoint-dir', CHECKPOINT_DIR,
    '--output-dir', OUT_DIR,
]
print('Run 1: All 4 models WITH L3, seed=42, 100Q')
print(f'CMD: {" ".join(cmd[2:])}\n')
result = subprocess.run(cmd, capture_output=False, text=True)
print(f'\n[{"OK" if result.returncode == 0 else "FAIL"}] Run 1 complete')

## Step 5 — Run 2: 8B WITH L3, seed 123

In [ ]:
import subprocess, sys, os
os.chdir(REPO_DIR)

cmd = [
    sys.executable, '-m', 'csam_project.benchmarks.benchmark_musique',
    '--provider', 'groq',
    '--model', 'llama-3.1-8b-instant',
    '--questions', str(N_QUESTIONS),
    '--dataset', DATASET,
    '--seed', '123',
    '--checkpoint-dir', CHECKPOINT_DIR,
    '--output-dir', OUT_DIR,
]
print('Run 2: Llama-3.1-8B WITH L3, seed=123, 100Q')
result = subprocess.run(cmd, capture_output=False, text=True)
print(f'\n[{"OK" if result.returncode == 0 else "FAIL"}] Run 2 complete')

## Step 6 — Run 3: 8B WITH L3, seed 456

In [ ]:
import subprocess, sys, os
os.chdir(REPO_DIR)

cmd = [
    sys.executable, '-m', 'csam_project.benchmarks.benchmark_musique',
    '--provider', 'groq',
    '--model', 'llama-3.1-8b-instant',
    '--questions', str(N_QUESTIONS),
    '--dataset', DATASET,
    '--seed', '456',
    '--checkpoint-dir', CHECKPOINT_DIR,
    '--output-dir', OUT_DIR,
]
print('Run 3: Llama-3.1-8B WITH L3, seed=456, 100Q')
result = subprocess.run(cmd, capture_output=False, text=True)
print(f'\n[{"OK" if result.returncode == 0 else "FAIL"}] Run 3 complete')

## Step 7 — Run 4: 8B WITHOUT L3, seed 42 (L3 ablation)
Disabling L3 isolates the knowledge-graph multi-hop contribution.
Delta vs Run 1 (8B) = value of L3 for multi-hop QA (PB-13).

In [ ]:
import subprocess, sys, os
os.chdir(REPO_DIR)

cmd = [
    sys.executable, '-m', 'csam_project.benchmarks.benchmark_musique',
    '--provider', 'groq',
    '--model', 'llama-3.1-8b-instant',
    '--questions', str(N_QUESTIONS),
    '--dataset', DATASET,
    '--seed', '42',
    '--no-l3',
    '--checkpoint-dir', CHECKPOINT_DIR,
    '--output-dir', OUT_DIR,
]
print('Run 4: Llama-3.1-8B WITHOUT L3 (ablation), seed=42, 100Q')
result = subprocess.run(cmd, capture_output=False, text=True)
print(f'\n[{"OK" if result.returncode == 0 else "FAIL"}] Run 4 complete')

## Step 8 — Results summary + L3 contribution delta

In [ ]:
import json, os, glob

files = sorted(glob.glob(os.path.join(OUT_DIR, 'results_musique_*.json')))

print('=' * 80)
print('MUSIQUE RESULTS')
print('=' * 80)
print(f'{"File":<58} {"Avg F1":>8} {"Sem":>8} {"N":>5}')
print('-' * 80)

data = {}
for fp in files:
    with open(fp) as f: d = json.load(f)
    name = os.path.basename(fp)[:57]
    f1   = d.get('avg_f1', d.get('micro_f1', 0))
    sem  = d.get('avg_semantic_sim', 0)
    n    = d.get('num_questions', 0)
    print(f'{name:<58} {f1:>8.4f} {sem:>8.4f} {n:>5}')
    data[os.path.basename(fp)] = {'f1': f1, 'sem': sem}

# L3 contribution delta (8B with vs without)
l3_key    = next((k for k in data if 'llama-3.1-8b' in k and 's42' in k and 'nol3' not in k), None)
nol3_key  = next((k for k in data if 'nol3' in k), None)
if l3_key and nol3_key:
    delta = data[l3_key]['f1'] - data[nol3_key]['f1']
    print(f'\nL3 hop contribution (8B seed=42): {delta:+.4f} F1')
    print(f'  With L3:    {data[l3_key]["f1"]:.4f}')
    print(f'  Without L3: {data[nol3_key]["f1"]:.4f}')

print(f'\nTotal result files: {len(files)}')

## Step 9 — Save / Download results

In [ ]:
import shutil, os, glob

all_files = glob.glob(os.path.join(OUT_DIR, '*.json'))

if os.path.exists('/kaggle'):
    kaggle_out = '/kaggle/working/nb3_musique'
    os.makedirs(kaggle_out, exist_ok=True)
    for fp in all_files:
        dest = os.path.join(kaggle_out, os.path.basename(fp))
        shutil.copy(fp, dest)
        print(f'Kaggle output: {dest}')
else:
    try:
        from google.colab import files
        for fp in all_files:
            files.download(fp)
            print(f'Downloaded: {fp}')
    except ImportError:
        print('Files saved at:')
        for fp in sorted(all_files): print(f'  {fp}')